The primary goal of the Product Portfolio Analysis is to strategically assess our product lineup to make data-driven decisions on investment, marketing, and product development. By categorising products into Stars, Cash Cows, Question Marks, or Dogs, This aim to:
- Maximise Profitability: Focus on products that contribute most to your bottom line.
- Optimise Resource Allocation: Direct resources and marketing efforts towards products with the highest growth potential.
- Innovate and Evolve: Identify areas for innovation and product development to stay competitive and meet customer demands.

In [1]:
import pandas as pd

# Assuming you've set your project and dataset
project_id = 'jr-data-training'
dataset_id = 'dbt_cafeanalytics'

# Function to query BigQuery and return a DataFrame
def query_bigquery(sql):
    return pd.read_gbq(sql, project_id=project_id, progress_bar_type=None)

# List of tables you want to query, replace this with dynamic fetching if needed
tables = ['paymentStatusDim', 'productDim', 'timeDim', 'openingHoursDim', 'customerDim', 'fact_orders', 'fact_product_sales']  
dfs = {}

# Loop through tables and query each, storing each in the dfs dictionary
for table in tables:
    sql = f"SELECT * FROM `{project_id}.{dataset_id}.{table}`"
    dfs[table] = query_bigquery(sql)
    # print(f"Data from {table}:", dfs[table].head(), "\n")  # Just printing the first few rows for each table as an example


In [2]:
# Accessing individual DataFrames
# For example, to work with the paymentStatusDim DataFrame:
payment_status_df = dfs['paymentStatusDim']
timeDim_df = dfs['timeDim']
productDim_df = dfs['productDim']
openingHoursDim_df = dfs['openingHoursDim']
customerDim_df = dfs['customerDim']
fact_orders_df = dfs['fact_orders']
fact_product_sales_df = dfs['fact_product_sales']



- customerDim and fact_orders: Can be merged on customer_id to enrich orders with customer information.
- productDim and fact_product_sales: Merge on item_name to get detailed product information for each sale.
- timeDim: Merge with fact_orders and fact_product_sales on the date fields (date_created in sales tables and date_key in timeDim after formatting if necessary).
- openingHoursDim: This doesn't directly merge with sales data unless you're analyzing daily aggregates or opening hours' impact on sales.
- paymentStatusDim and fact_orders: Merge on status to enrich orders with readable status descriptions.


In [3]:
# Merge customerDim with fact_orders: Ensure that 'customer_id' is of the same type in both DataFrames
fact_orders_df['customer_id'] = fact_orders_df['customer_id'].astype(str)
customerDim_df['customer_id'] = customerDim_df['customer_id'].astype(str)
# Merge
orders_with_customer_info = pd.merge(fact_orders_df, customerDim_df, on='customer_id', how='left')



# Merge productDim with fact_product_sales: Merge on 'item_name'
sales_with_product_info = pd.merge(fact_product_sales_df, productDim_df, on='item_name', how='left')



# Example conversion, adjust formats as needed
fact_product_sales_df['date_key'] = pd.to_datetime(fact_product_sales_df['date_created']).dt.date
timeDim_df['date_key'] = pd.to_datetime(timeDim_df['date_key']).dt.date
# Merge
sales_with_time_info = pd.merge(fact_product_sales_df, timeDim_df, on='date_key', how='left')




# Merge paymentStatusDim with fact_orders: Ensure 'status' is of the same type
fact_orders_df['status'] = fact_orders_df['status'].astype(str)
payment_status_df['status'] = payment_status_df['status'].astype(str)
# Merge
orders_with_payment_status = pd.merge(fact_orders_df, payment_status_df, on='status', how='left')

perform analyses like 
- customer segmentation, 
- product performance review, 
- sales trends over time, and the 
- impact of operational hours on sales. 
- identifying your most valuable customers,
- top-selling products, or trends that inform operational decisions.

In [4]:
# Now you have a DataFrame that includes daily sales and corresponding operating hours information.
# You can filter this DataFrame to analyze days with special notes, comparing sales on these days vs. regular operating days.

# Aggregate fact_product_sales_df to daily sales
daily_sales = fact_product_sales_df.groupby(fact_product_sales_df['date_created'].dt.date).agg(total_daily_sales=('total_sales', 'sum')).reset_index()
# Convert 'date_created' in daily_sales to datetime, if not already done
daily_sales['date_created'] = pd.to_datetime(daily_sales['date_created'])
# Ensure 'actual_date' in openingHoursDim_df is also datetime
openingHoursDim_df['actual_date'] = pd.to_datetime(openingHoursDim_df['actual_date'])
# Now attempt the merge again
daily_sales_with_hours = pd.merge(daily_sales, openingHoursDim_df, left_on='date_created', right_on='actual_date', how='left')



How openingHoursDim Could Be Used:


Analyzing Sales Performance Relative to Operating Hours: By understanding when your café is open, including any special notes about closures or reduced hours, you can analyze sales data to see how performance varies on days with standard vs. non-standard hours. This requires aggregating sales data to a daily level to match the granularity of openingHoursDim.

Impact of Special Hours on Sales: Special hours or closures (like holidays) might have a significant impact on sales. By identifying these days using openingHoursDim and comparing sales on these days to your average, you can gauge the impact.

Operational Planning: Insights from combining sales data with operating hours can inform staffing and inventory decisions. For example, if sales significantly increase on days before a special closure, you might plan accordingly.

In [ ]:
""" from ydata_profiling import ProfileReport
# distribution of order values across different customer segments, identify frequent customers, and analyze the range of order values and volumes.
# For orders_with_customer_info DataFrame
profile_orders_customers = ProfileReport(orders_with_customer_info, title="Orders with Customer Info Profile")
profile_orders_customers.to_file("orders_customers_profile.html") 
# or use .to_widget() to save the report as an HTML file

profile_orders_products = ProfileReport(sales_with_product_info, title="Orders with Product Info Profile")
profile_orders_products.to_file("orders_products_profile.html") 

profile_orders_payments = ProfileReport(orders_with_payment_status, title="Orders with Payment Info Profile")
profile_orders_payments.to_file("orders_payments_profile.html") 

profile_daily_sales_with_hours = ProfileReport(daily_sales_with_hours, title="daily_sales_with_hours Info Profile")
profile_daily_sales_with_hours.to_file("profile_daily_sales_with_hours_profile.html") 

# Assuming df is your DataFrame
sales_with_time_info['date_created'] = pd.to_datetime(sales_with_time_info['date_created'])
sales_with_time_info['date_key'] = pd.to_datetime(sales_with_time_info['date_key'])

profile_orders_sales = ProfileReport(sales_with_time_info, title="Orders with Sales Info Profile")
profile_orders_sales.to_file("orders_sales_profile.html") 

"""